In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# print(1)

from enum import Enum
import sys
from typing import Union, Optional
from loguru import logger as loguru_logger
from pathlib import Path


class LogFormat(str, Enum):
    DEFAULT = "<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}"
    ALTERNATIVE = "<level>{time:HH:mm:ss}</level> | <level>{message}</level>"
    DETAILED = "<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{line}</cyan> | {message}"
    BRACKETED = "[<green>{time:HH:mm}</green>] [<level>{level}</level>] {message}"


class LogMode(str, Enum):
    DEV = "dev"  # Console + detailed output
    PROD = "prod"  # File-based, minimal console
    JUPYTER = "jupyter"  # Optimized for Jupyter cell output
    CUSTOM = "custom"  # Fully manual configuration


def setup_logger(
        logger=loguru_logger,
        level: str = "INFO",
        format: Union[LogFormat, str] = LogFormat.DEFAULT,
        mode: LogMode = LogMode.DEV,
        console: bool = True,
        file: Optional[Union[str, Path]] = None,
        jupyter: bool = False,
        rotation: Optional[str] = None,  # e.g., "1 MB" or "00:00" for daily
        retention: Optional[str] = None,  # e.g., "7 days"
        colorize: bool = True,
):
    """
    Configure the loguru logger with flexible output options.

    Args:
        logger: The loguru logger instance to configure.
        level: Logging level (e.g., "INFO", "DEBUG").
        format: Log format (from LogFormat enum or custom string).
        mode: Predefined mode ("dev", "prod", "jupyter", "custom").
        console: Enable console output (sys.stderr by default).
        file: Path to log file (None disables file logging).
        jupyter: Enable Jupyter-specific output (sys.stdout).
        rotation: File rotation policy (e.g., "1 MB", "daily").
        retention: File retention policy (e.g., "7 days").
        colorize: Enable colorized output where supported.
    """
    logger.remove()  # Clear existing handlers

    # Adjust defaults based on mode
    if mode == LogMode.DEV:
        console = True
        jupyter = False
        file = None
        format = LogFormat.DETAILED
    elif mode == LogMode.PROD:
        console = True
        jupyter = False
        file = file or "app.log"
        format = LogFormat.DEFAULT
        rotation = rotation or "1 MB"
        retention = retention or "7 days"
    elif mode == LogMode.JUPYTER:
        console = False
        jupyter = True
        file = None
        format = LogFormat.DEFAULT

    # Add console sink (sys.stderr)
    if console:
        logger.add(
            sink=sys.stderr,
            format=format,
            level=level,
            colorize=colorize,
        )

    # Add Jupyter sink (sys.stdout)
    if jupyter:
        logger.add(
            sink=sys.stdout,
            format=format,
            level=level,
            colorize=colorize,
        )

    # Add file sink
    if file:
        logger.add(
            sink=Path(file),
            format=format,
            level=level,
            colorize=False,  # Files don’t need color
            rotation=rotation,
            retention=retention,
        )

    return logger


logger = setup_logger(jupyter=True)

logger.info("Hello, world!")

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
"""
LLM Utils Test Router

This file tests the behavior of LLM for parsing structured data with optional fields.
"""

from aiogram import Router
from aiogram.filters import Command
from aiogram.types import Message
from botspot import commands_menu
from botspot.utils import send_safe, send_typing_status
from pydantic import BaseModel, Field
from typing import Optional, List

from _app import App

# Create router and app
router = Router()
app = App()


# Define test models with optional and required fields
class Person(BaseModel):
    name: str = Field(description="The person's full name")
    age: Optional[int] = Field(None, description="The person's age in years")
    email: str = Field(description="The person's email address")
    interests: Optional[List[str]] = Field(None,
                                           description="The person's interests or hobbies")



In [ ]:
@commands_menu.botspot_command("form_test", "Test LLM with forms")
@router.message(Command("form_test"))
async def form_test_handler(message: Message):
    """Test if LLM can differentiate between optional and required fields."""
    from botspot import get_dependency_manager
    deps = get_dependency_manager()
    logger.info("Starting form_test_handler")
    await send_safe(
        message.chat.id,
        "Testing LLM with a form that has both required and optional fields..."
    )

    await send_typing_status(message)

    # First, use complete but minimal information
    test_prompt = "Name: John Doe, Email: john@example.com"
    logger.info(f"Testing with minimal info: {test_prompt}")

    try:
        await send_safe(message.chat.id, f"Testing with minimal info: {test_prompt}")

        result = await deps.llm_provider.aquery_llm_structured(
            prompt=f"Parse this into a person record: {test_prompt}",
            output_schema=Person,
            user=str(message.from_user.id)
        )

        # Format the result
        interests_text = ", ".join(
            result.interests) if result.interests else "None provided"
        await send_safe(
            message.chat.id,
            f"Result with minimal info:\n"
            f"• Name: {result.name}\n"
            f"• Age: {result.age if result.age is not None else 'None provided'}\n"
            f"• Email: {result.email}\n"
            f"• Interests: {interests_text}"
        )
        logger.info(f"Result with minimal info: {result}")
    except Exception as e:
        logger.error(f"Error in minimal info test: {e}")
        await send_safe(message.chat.id, f"❌ Error with minimal info: {str(e)}")

    # Second, use missing required information
    test_prompt = "Age: 30, Interests: reading, hiking"
    logger.info(f"Testing with missing required fields: {test_prompt}")

    try:
        await send_safe(message.chat.id,
                        f"Testing with missing required fields: {test_prompt}")
        result = await deps.llm_provider.aquery_llm_structured(
            prompt=f"Parse this into a person record: {test_prompt}",
            output_schema=Person,
            user=str(message.from_user.id)
        )

        # Format the result
        interests_text = ", ".join(
            result.interests) if result.interests else "None provided"
        await send_safe(
            message.chat.id,
            f"Result with missing required fields:\n"
            f"• Name: {result.name}\n"
            f"• Age: {result.age if result.age is not None else 'None provided'}\n"
            f"• Email: {result.email}\n"
            f"• Interests: {interests_text}"
        )
        logger.info(f"Result with missing required fields: {result}")
    except Exception as e:
        logger.error(f"Error in missing required fields test: {e}")
        await send_safe(
            message.chat.id,
            f"❌ Error with missing required fields: {str(e)}\n\n"
            f"This shows the LLM doesn't automatically handle missing required fields."
        )

    # Third, test with irrelevant information to see what happens
    test_prompt = "Weather: sunny, Location: New York, Date: 2025-03-19"
    logger.info(f"Testing with irrelevant information: {test_prompt}")

    try:
        await send_safe(message.chat.id,
                        f"Testing with irrelevant information: {test_prompt}")

        result = await deps.llm_provider.aquery_llm_structured(
            prompt=f"Parse this into a person record: {test_prompt}",
            output_schema=Person,
            user=str(message.from_user.id)
        )

        # Format the result
        interests_text = ", ".join(
            result.interests) if result.interests else "None provided"
        await send_safe(
            message.chat.id,
            f"Result with irrelevant info:\n"
            f"• Name: {result.name}\n"
            f"• Age: {result.age if result.age is not None else 'None provided'}\n"
            f"• Email: {result.email}\n"
            f"• Interests: {interests_text}"
        )
        logger.info(f"Result with irrelevant info: {result}")
    except Exception as e:
        logger.error(f"Error in irrelevant info test: {e}")
        await send_safe(
            message.chat.id,
            f"❌ Error with irrelevant info: {str(e)}\n\n"
            f"This shows the LLM struggles with completely irrelevant input."
        )

    # Summary
    await send_safe(
        message.chat.id,
        "📝 Summary of form parsing test:\n\n"
        "1. The LLM can handle optional fields when they're missing\n"
        "2. It requires all required fields (throws error otherwise)\n"
        "3. It struggles with completely irrelevant input\n\n"
        "This means we need to build a custom solution to handle missing required fields "
        "and communicate when information is missing or irrelevant."
    )
    logger.info("Completed form_test_handler")


In [ ]:
# Initial setup (from your earlier code)
from pathlib import Path
from aiogram import Bot, Dispatcher
from aiogram.client.default import DefaultBotProperties
from aiogram.enums import ParseMode
from botspot.core.bot_manager import BotManager
from dotenv import load_dotenv
from loguru import logger
import asyncio
import nest_asyncio

nest_asyncio.apply()
load_dotenv(Path.cwd() / ".env")

# router = Router()
# from router import app, router

dp = Dispatcher()
dp.include_router(router)

bot = Bot(
    token=app.config.telegram_bot_token,
    default=DefaultBotProperties(parse_mode=ParseMode.HTML)
)


async def start_bot():
    setup_logger(logger)
    bm = BotManager(bot=bot)
    bm.setup_dispatcher(dp)
    await dp.start_polling(bot)


# Start the bot
asyncio.create_task(start_bot())
print("Bot is running!")

In [ ]:
async def stop_bot():
    await dp.stop_polling()


# Run this in a separate cell to stop the bot
await stop_bot()

In [ ]:
import importlib
import botspot

# import router

importlib.reload(botspot)
# from router import router  # Re-import the updated router

# # Re-include the updated router in the dispatcher (if needed)
# dp.include_router(router)
# print("Router reloaded and re-included!")

In [ ]:

from litellm import completion
from pydantic import BaseModel

# add to env var
# os.environ["OPENAI_API_KEY"] = ""

messages = [{"role": "user", "content": "List 5 important events in the XIX century"}]


class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


class EventsList(BaseModel):
    events: list[CalendarEvent]


resp = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
    response_format=EventsList
)

print("Received={}".format(resp))

In [ ]:
resp.__dict__


In [ ]:
type(resp.choices[0].message.content)

In [ ]:
resp.choices[0]